# Sentiment Classification of Microblog Text

**Liqi Liang** | Penn Machine Learning Project(MSSP 6080)

This project explores sentiment classification of microblog posts using a bag-of-words approach. Using a labeled dataset of 5,000 tweets, I compare multiple machine learning classifiers — Naïve Bayes, Logistic Regression, and SVMs — across different feature spaces (unigrams and N-grams) and regularization strategies. Model selection is based on 10-fold cross-validation optimized for Cohen's Kappa, followed by final evaluation on a held-out test set.

## Setup


In [ ]:
import pandas as pd
import math
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib import dates
from datetime import datetime
import re
import calendar
import json

from sklearn.metrics import accuracy_score, precision_score, recall_score, cohen_kappa_score, confusion_matrix, f1_score, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import BernoulliNB, ComplementNB, GaussianNB, MultinomialNB
from sklearn import tree
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer

import spacy
from spacy.lang.en import English

import jieba


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Data Preparation

The dataset consists of 5,000 labeled microblog posts, each tagged with two sentiment labels (`labels_A` and `labels_B`). After removing null values, the data is split into an 80% training set (4,000 examples) and a 20% held-out test set (1,000 examples).

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/MSSP 6080/training_labeled_sampled_5k.csv")
df = df.dropna()
df.head()

Index(['text', 'labels_A', 'labels_B'], dtype='object')

In [ ]:
df_training = df.sample(frac=0.8, random_state=333)
df_testing = df.drop(df_training.index)

print("Training set size:", len(df_training))
print("Test set size:", len(df_testing))

Length of training set: 4000
Length of test set: 1000


## 2. Classifier Evaluation Framework

I define reusable functions for cross-validated model comparison. All models are evaluated using 10-fold stratified cross-validation on the training set, optimizing for Cohen's Kappa as the primary metric.

In [ ]:
# Start by defining a function to evaluate a classifier's predictions
def evaluate(y_pred, y_actual, metrics, model_name = 'model'):
    # Compute Confusion Matrix
    conf_matrix = confusion_matrix(y_actual, y_pred)

    # Compute and store each metric
    model_metrics = {}
    for (metric_name, metric) in metrics.items():
        result = metric(y_actual, y_pred)
        model_metrics[metric_name] = result

    return conf_matrix, model_metrics
def evaluate_one_fold(classifier_name, classifier, X_train, y_train, X_test, y_test, metrics, fold_num, noisy = 'loud', labels=[]):
    model = classifier.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    conf_matrix, model_metrics = evaluate(y_pred, y_test, metrics, model_name = classifier_name)
    if noisy == 'quiet' and fold_num == 0:
        print(f"{classifier_name}: Fold {fold_num}", end = '')
    elif noisy == 'quiet':
        print(f'...{fold_num}', end ='')
    elif noisy == 'loud':
        print(f"{classifier_name}: Fold {fold_num} Results")
        ConfusionMatrixDisplay(conf_matrix, labels).plot(values_format='.4g')
        plt.show()
        print(model_metrics)
        print("------------------------")

    return model_metrics
def evaluate_all_folds(classifier_name, classifier, X, y, kf, metrics, noisy = 'loud', labels=[]):
    all_fold_metrics = {metric_name: [] for metric_name in metrics}
    for fold_num, (train_index, test_index) in enumerate(kf.split(X, y)):
        X_train = X.iloc[train_index]
        X_test = X.iloc[test_index]
        y_train = y.iloc[train_index]
        y_test = y.iloc[test_index]
        model_metrics =  evaluate_one_fold(classifier_name, classifier, X_train, y_train, X_test, y_test, metrics, fold_num, noisy, labels=labels)
        [all_fold_metrics[metric_name].append(metric_val) for metric_name, metric_val in model_metrics.items()]

    return all_fold_metrics
def compare_classifiers(classifiers, metrics, metric_to_optimize, df, feature_set,
                        target, folds = 10, shuffle = True, noisy='loud', labels=[]):
    best = 0
    best_name = None
    classifier_comparison = {}
    X = df.loc[:, feature_set]
    X = pd.get_dummies(X)
    y = df[target]
    kf = StratifiedKFold(n_splits=folds, shuffle=shuffle)
    for classifier_name, classifier in classifiers.items():
        # Evaluate on all metrics for all folds
        all_fold_metrics = evaluate_all_folds(classifier_name, classifier, X, y, kf, metrics, noisy = noisy, labels=labels)
        optimization_metric_avg = np.mean(all_fold_metrics[metric_to_optimize])
        if optimization_metric_avg > best:
            best = optimization_metric_avg
            best_name = classifier_name
        classifier_comparison[classifier_name] = all_fold_metrics
        if noisy == 'quiet':
            print()
            print(f"Average {metric_to_optimize}: {optimization_metric_avg:.3f}")
            print('-------------')
    return best, best_name, classifier_comparison

## 3. Feature Engineering

Text is represented using a bag-of-words (BOW) model. I construct a vocabulary of the top 1,000 most frequent tokens using `CountVectorizer`, and also implement an N-gram helper function to support unigram vs. bigram comparisons.

In [ ]:
vocab_size = 1000
vectorizer = CountVectorizer(max_features=vocab_size)
X = vectorizer.fit_transform(df_training['text'])

In [ ]:
bow_df = pd.DataFrame(X.toarray())
column_names = [str(i) for i in range(vocab_size)]
for k, v in vectorizer.vocabulary_.items():
    column_names[v] = k
bow_df.columns = column_names
bow_df["labels_A"] = df_training["labels_A"].values
bow_df["labels_B"] = df_training["labels_B"].values
bow_df.head()

,10,100,11,12,16,1st,20,2day,2nd,30,...,yet,you,young,your,yours,youtube,yr,yup,labels_A,labels_B
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Negative,Negative
1,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,Negative,Negative
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Positive,Positive
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Negative,Negative
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Negative,Negative
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Positive,Positive
3996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Positive,Positive
3997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,Negative,Negative
3998,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,Positive,Positive


In [ ]:
def ngrams(df, vocab_size = 1000, max_n=1):
  vectorizer = CountVectorizer(max_features=vocab_size, ngram_range=(1,max_n))
  X = vectorizer.fit_transform(df["text"])

  bow_df = pd.DataFrame(X.toarray())
  column_names = [str(i) for i in range(vocab_size)]
  for k, v in vectorizer.vocabulary_.items():
    column_names[v] = k
  bow_df.columns = column_names

  bow_df["labels_A"] = df["labels_A"].reset_index()['labels_A']
  return bow_df
# create unigram and bigrams DataFrame
unigram_df = ngrams(df_training, max_n=1)
bigram_df = ngrams(df_training, max_n=2)

## 4. Model Comparison and Selection

I evaluate classifiers across five experimental conditions using 10-fold stratified cross-validation, optimizing for Cohen's Kappa. Each experiment isolates a specific modeling choice.

### 4a. Naïve Bayes vs. Logistic Regression vs. SVM (Unigram Features)


In [ ]:
# Pick Classifiers to Compare
from sklearn.svm import LinearSVC, SVC
classifiers = {
    "Bernoulli NB": BernoulliNB(),
    "Linear SVM": LinearSVC(),
    "RBF SVM": SVC(kernel='rbf'),
    "Poly SVM": SVC(kernel='poly'),
    "Complement NB": ComplementNB(),
    "Multinomial NB": MultinomialNB(),
    "Logistic Regression": LogisticRegression(penalty=None, solver="lbfgs", multi_class='ovr', max_iter=10000, random_state=123),
}

# Set a list of metrics we want to use to compare our classifiers
metrics = {
    "Accuracy" : lambda y,y_pred: 100*accuracy_score(y,y_pred),
    "Kappa"    : cohen_kappa_score
}

# Choose a metric to optimize over
metric_to_optimize = 'Kappa'

# Pick features to use
unigram_features = [col for col in bow_df.columns if col not in ["labels_A", "labels_B"]]
feature_set = unigram_features

# Compare models and display final result
sorted_sentiments = ["Negative", "Positive"]

best_a, best_name_a, classifier_comparison_a = compare_classifiers(
    classifiers, metrics, metric_to_optimize,
    bow_df, feature_set, "labels_A",
    noisy='quiet', labels=sorted_sentiments
)

print(f"Best classifier is: {best_name_a} \nWith K={best_a:.3f}.")

Bernoulli NB: Fold 0...1...2...3...4...5...6...7...8...9
Average Kappa: 0.437
-------------
Linear SVM: Fold 0...1...2...3...4...5...6...7...8...9
Average Kappa: 0.378
-------------
RBF SVM: Fold 0...1...2...3...4...5...6...7...8...9
Average Kappa: 0.411
-------------
Poly SVM: Fold 0...1...2...3...4...5...6...7...8...9
Average Kappa: 0.263
-------------
Complement NB: Fold 0...1...2...3...4...5...6...7...8...9
Average Kappa: 0.430
-------------
Multinomial NB: Fold 0...1...2...3...4...5...6...7...8...9
Average Kappa: 0.426
-------------


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


Logistic Regression: Fold 0

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


...1

...

...7

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


...8

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


...9
Average Kappa: 0.351
-------------
Best classifier is: Bernoulli NB 
With K=0.437.


**Result:** The best-performing model on the unigram feature space was **Bernoulli NB**, achieving a Kappa score of **0.437** based on 10-fold cross-validation.


### 4b. Unigram vs. Bigram Feature Space


In [ ]:
# Pick Classifiers to Compare
from sklearn.svm import LinearSVC, SVC
classifiers = {
     "Logistic Regression": LogisticRegression(penalty=None, solver="lbfgs", multi_class='ovr', max_iter=10000, random_state=123)
}

### Compare classifiers on unigrams ###
# Pick features to use
feature_set_1 = list(unigram_df.columns[:-1])
sorted_sentiments = ["Negative", "Positive"]

# Compare models and display final result
best, best_name, classifier_comparison = compare_classifiers(
    classifiers, metrics, metric_to_optimize,
    unigram_df, feature_set_1, "labels_A",
    labels= sorted_sentiments, noisy='quiet'
)

### Compare classifiers on bigrams ###
# Pick features to use
feature_set_2 = list(bigram_df.columns[:-1])

# Compare models and display final result
best, best_name, classifier_comparison = compare_classifiers(
    classifiers, metrics, metric_to_optimize,
    bigram_df, feature_set_2, "labels_A",
    labels=sorted_sentiments, noisy='quiet'
)

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


Logistic Regression: Fold 0

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


...1

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


...

...7

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


...8

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


...9
Average Kappa: 0.340
-------------


**Result:** Logistic Regression on unigram features (Kappa = **0.364**) outperformed bigrams (Kappa = 0.340), suggesting that single-word features capture more signal for this task.


### 4c. Naïve Bayes Variants: BernoulliNB vs. ComplementNB vs. MultinomialNB


In [ ]:
# Pick Classifiers to Compare
classifiers = {
    "Bernoulli NB": BernoulliNB(),
    "Complement NB": ComplementNB(),
    "Multinomial NB": MultinomialNB()
}

# Set a list of metrics we want to use to compare our classifiers
metrics = {
    "Accuracy" : lambda y,y_pred: 100*accuracy_score(y,y_pred),
    "Kappa"    : cohen_kappa_score
}

# Choose a metric to optimize over
metric_to_optimize = 'Kappa'

# Pick features to use
bow_features = column_names
feature_set = bow_features

# Compare models and display final result
best_i, best_name_i, classifier_comparison_i = compare_classifiers(
    classifiers, metrics, metric_to_optimize,
    bow_df, feature_set, "labels_A",
    labels=sorted_sentiments, noisy='quiet'
    )

print(f"Best classifier is: {best_name_i} \nWith K={best_i:.3f}.")

Bernoulli NB: Fold 0...1...2...3...4...5...6...7...8...9
Average Kappa: 0.446
-------------
Complement NB: Fold 0...1...2...3...4...5...6...7...8...9
Average Kappa: 0.427
-------------
Multinomial NB: Fold 0...1...2...3...4...5...6...7...8...9
Average Kappa: 0.426
-------------
Best classifier is: Bernoulli NB 
With K=0.446.


**Result:** **Bernoulli NB** achieved the highest Kappa of **0.446**, outperforming Complement NB (0.427) and Multinomial NB (0.426).


### 4d. SVM Kernel Comparison: Linear vs. RBF vs. Polynomial


In [ ]:
# Pick Classifiers to Compare
classifiers = {
    "Linear SVM": SVC(kernel='linear'),
    "RBF SVM": SVC(kernel='rbf'),
    "Poly SVM": SVC(kernel='poly')
}

# Set a list of metrics we want to use to compare our classifiers
metrics = {
    "Accuracy" : lambda y,y_pred: 100*accuracy_score(y,y_pred),
    "Kappa"    : cohen_kappa_score
}

# Choose a metric to optimize over
metric_to_optimize = 'Kappa'

# Pick features to use
feature_set = bow_features

# Compare models and display final result
best, best_name, classifier_comparison = compare_classifiers(
    classifiers, metrics, metric_to_optimize,
    bow_df, feature_set, "labels_A",
    labels=sorted_sentiments, noisy='quiet'
)

print(f"Best SVM kernel is: {best_name} \nWith K={best:.3f}.")

Linear SVM: Fold 0...1...2...3...4...5...6...7...8...9
Average Kappa: 0.383
-------------
RBF SVM: Fold 0...1...2...3...4...5...6...7...8...9
Average Kappa: 0.414
-------------
Poly SVM: Fold 0...1...2...3...4...5...6...7...8...9
Average Kappa: 0.249
-------------
Best SVM kernel is: RBF SVM 
With K=0.414.


**Result:** The **RBF kernel** produced the best SVM performance with Kappa = **0.414**, followed by Linear (0.383) and Polynomial (0.249).


### 4e. Logistic Regression: L1 vs. L2 vs. No Regularization


In [ ]:
# Pick Classifiers to Compare
classifiers = {
    "LogisticRegression l2": LogisticRegression(penalty='l2', solver='saga', max_iter=10000, random_state=123),
    "LogisticRegression l1": LogisticRegression(penalty='l1', solver='saga', max_iter=10000, random_state=123),
    "Unregularized Features": LogisticRegression(penalty=None, solver='saga', max_iter=10000, random_state=123)
}
# Set a list of metrics we want to use to compare our classifiers
metrics = {
    "Accuracy" : lambda y,y_pred: 100*accuracy_score(y,y_pred),
    "Kappa"    : cohen_kappa_score
}

# Choose a metric to optimize over
metric_to_optimize = 'Kappa'

# Pick features to use
feature_set = bow_features

# Compare models and display final result
best, best_name, classifier_comparison = compare_classifiers(
    classifiers, metrics, metric_to_optimize,
    bow_df, feature_set, "labels_A",
    labels=sorted_sentiments, noisy='quiet'
)

print(f"Best SVM kernel is: {best_name} \nWith K={best:.3f}.")

LogisticRegression l2: Fold 0...1...2...3...4...5...6...7...8...9
Average Kappa: 0.425
-------------
LogisticRegression l1: Fold 0...1...2...3...4...5...6...7...8...9
Average Kappa: 0.429
-------------
Unregularized Features: Fold 0...1...2...3...4...5...6...7...8...9
Average Kappa: 0.367
-------------
Best SVM kernel is: LogisticRegression l1 
With K=0.429.


**Result:** **L1 regularization** yielded the best Kappa of **0.429**, slightly outperforming L2 (0.425) and unregularized logistic regression (0.367).


## 5. Final Model Evaluation

Across all experiments, **Bernoulli Naïve Bayes with unigram features** consistently achieved the highest Kappa scores. I retrain this model on the full 80% training set and evaluate it on the held-out 20% test set.

In [ ]:
# Train a Naive Bayes classifier with unigram features.
# choose BernoulliNB() as our Naive Bayes classifer.

#set up unigram dataframe (for both test/training)
unigram_df=ngrams(df, max_n=1)
feature_set = list(unigram_df.columns[:-1])
unigram_training=unigram_df.iloc[df_training.index,:]
unigram_testing=unigram_df.iloc[df_testing.index,:]

#create your x_train,y_train,x_test,y_test
X_train=unigram_training.loc[:, feature_set]
y_train=unigram_training.iloc[:,-1]
X_test = unigram_testing.loc[:, feature_set]
y_test = unigram_testing.iloc[:, -1]

#set BenroulliNB as your classifier
NB=BernoulliNB()
NB.fit(X_train, y_train)

BernoulliNB()

In [ ]:
from sklearn.metrics import cohen_kappa_score, make_scorer
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score


# Calculate and print the estimated performance from 10-fold-validation
a_scores = cross_val_score(estimator = NB,
                           X = X_train,
                           y=y_train,
                           cv=10,
                           scoring='accuracy'
                           )
# print('10-fold accuracy: %s' % a_scores)
print("Estimated performance from cross-validation:")
print('10-fold accuracy: %.3f +/- %.3f' % (np.mean(a_scores), np.std(a_scores)))

kappa_scorer = make_scorer(cohen_kappa_score)
k_scores = cross_val_score(estimator = NB,
                           X = X_train,
                           y=y_train,
                           cv=10,
                           scoring=kappa_scorer
                           )
# print('10-fold Cohen's Kappa: %s' % k_scores)
print('10-fold Kappa: %.3f +/- %.3f' % (np.mean(k_scores), np.std(k_scores)))
print("------------------------------------")

# Calculate and print the performance of the Bernoulli NB model on the held-out test set
print("Performance on the held-out test set:")
y_pred = NB.predict(X_test)
accuracy_NB = accuracy_score(y_test, y_pred)
print(f"The accuracy is: {accuracy_NB:.4f}.")
kappa_NB = cohen_kappa_score(y_test, y_pred)
print(f"The Kappa is: {kappa_NB:.4f}.")

Estimated performance from cross-validation:
10-fold accuracy: 0.725 +/- 0.019
10-fold Kappa: 0.449 +/- 0.038
------------------------------------
Performance on the held-out test set:
The accuracy is: 0.7460.
The Kappa is: 0.4916.


In [ ]:
# Train the best performance model from part 1 of assignment (best trained = highest kappa)

# BernoulliNB is the best performance model
# Because the best-tuned model was also trained using unigram features, the same feature set can be reused without redefining X_train and X_test.
best_model = BernoulliNB() #same model
best_model.fit(X_train, y_train)

BernoulliNB()

In [ ]:
from sklearn.metrics import cohen_kappa_score, make_scorer
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score


# Calculate and print the estimated performance from 10-fold-validation
a_scores = cross_val_score(estimator = best_model,
                           X = X_train,
                           y=y_train,
                           cv=10,
                           scoring='accuracy'
                           )
# print('10-fold accuracy: %s' % a_scores)
print("Estimated performance from cross-validation:")
print('10-fold accuracy: %.3f +/- %.3f' % (np.mean(a_scores), np.std(a_scores)))

kappa_scorer = make_scorer(cohen_kappa_score)
k_scores = cross_val_score(estimator = best_model,
                           X = X_train,
                           y=y_train,
                           cv=10,
                           scoring=kappa_scorer
                           )
# print('10-fold Cohen's Kappa: %s' % k_scores)
print('10-fold Kappa: %.3f +/- %.3f' % (np.mean(k_scores), np.std(k_scores)))
print("------------------------------------")


# Calculate and print the performance of the best model model on the held-out test set.
print("Performance on the held-out test set(best_model - BernoulliNB):")
y_pred = best_model.predict(X_test)
accuracy_best_model = accuracy_score(y_test, y_pred)
print(f"The accuracy is: {accuracy_best_model:.4f}.")
kappa_best_model = cohen_kappa_score(y_test, y_pred)
print(f"The Kappa is: {kappa_best_model:.4f}.")

Estimated performance from cross-validation:
10-fold accuracy: 0.725 +/- 0.019
10-fold Kappa: 0.449 +/- 0.038
------------------------------------
Performance on the held-out test set(best_model - BernoulliNB):
The accuracy is: 0.7460.
The Kappa is: 0.4916.


## 6. Summary of Results

| Evaluation | Accuracy | Cohen's Kappa |
|---|---|---|
| Cross-validation estimate (BernoulliNB) | 72.50% | 0.449 |
| Held-out test set (BernoulliNB baseline) | 74.60% | 0.492 |
| Held-out test set (best-tuned model) | 74.60% | 0.492 |

The Bernoulli Naïve Bayes classifier with unigram features emerged as the best-performing model. Its Kappa of 0.492 on the held-out test set falls in the moderate agreement range (0.41–0.60), indicating consistent generalization beyond the training data. Notably, performance on the test set slightly exceeded cross-validation estimates, suggesting the model is not overfitting. The unigram feature space proved more informative than bigrams, and BernoulliNB's binary feature handling was better suited to sparse tweet-length text than its Multinomial or Complement counterparts.